In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

df = pd.read_csv('perf.csv')

section_cols = [c for c in df.columns if c not in ('fps', 'frame_time')]

for col in section_cols:
    df[col] = df[col].str.replace('ms', '').astype(float)

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(df))
bottom = np.zeros(len(df))

colors = plt.cm.tab10.colors
for i, col in enumerate(section_cols):
    ax.fill_between(x, bottom, bottom + df[col].values,
                    label=col, alpha=0.85, color=colors[i % len(colors)])
    bottom += df[col].values

ax.set_xlabel('Sample')
ax.set_ylabel('Time (ms)')
ax.set_title('Render Loop Timing Breakdown')
ax.legend(loc='upper right', fontsize=8)
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(axis='y', which='both', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import time
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms.functional as TF

from models.RecurrentDenoisingAutoencoder import RecurrentDenoisingAutoencoder

CHECKPOINT_PATH = Path("model_output_recurrent/autoencoder_best.pt")
EVAL_FOLDER     = "dataset_recurrent/eval"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
IN_CHANNELS  = ckpt["in_channels"]
OUT_CHANNELS = ckpt["out_channels"]
BASE         = ckpt["base_channels"]
SEQ_LEN      = ckpt["seq_len"]
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  eval_loss={ckpt['eval_loss']:.6f}")

model = RecurrentDenoisingAutoencoder(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    base=BASE,
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


class FullFrameEvalDataset(Dataset):
    def __init__(self, folder, seq_len):
        self.seq_len = seq_len
        self.sequences = sorted(Path(folder).glob("*/"))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq_dir = self.sequences[idx]

        color_frames  = sorted((seq_dir / "input").glob("*.png"))[:self.seq_len]
        depth_files   = sorted((seq_dir / "depth").glob("*.npy"))[:self.seq_len]
        target_frames = sorted((seq_dir / "target").glob("*.png"))[:self.seq_len]

        colors = [TF.to_tensor(Image.open(f).convert("RGB")) for f in color_frames]
        depths = [torch.from_numpy(np.load(f)).unsqueeze(0) for f in depth_files]

        xs = torch.stack([torch.cat([c, d], dim=0) for c, d in zip(colors, depths)])
        ys = torch.stack([TF.to_tensor(Image.open(f).convert("RGB")) for f in target_frames])
        return xs, ys


eval_loader = DataLoader(
    FullFrameEvalDataset(EVAL_FOLDER, SEQ_LEN),
    batch_size=1,
    shuffle=False,
    num_workers=0,
)


def zero_hidden(batch_size, base, H, W, device):
    return (
        torch.zeros(batch_size, base,     H,      W,      device=device),
        torch.zeros(batch_size, base * 2, H // 2, W // 2, device=device),
        torch.zeros(batch_size, base * 4, H // 4, W // 4, device=device),
        torch.zeros(batch_size, base * 8, H // 8, W // 8, device=device),
    )


frame_times = []

with torch.no_grad():
    for xs, ys in tqdm(eval_loader, desc="Evaluating"):
        xs = xs.to(device, non_blocking=True)

        B, T, _, H, W = xs.shape
        h1, h2, h3, h4 = zero_hidden(B, BASE, H, W, device)

        for t in range(T):
            x_t = xs[:, t]

            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            _, h1, h2, h3, h4 = model(x_t, h1, h2, h3, h4)

            if device.type == "cuda":
                torch.cuda.synchronize()
            frame_times.append((time.perf_counter() - t0) * 1000)

        h1, h2, h3, h4 = (h.detach() for h in (h1, h2, h3, h4))

total_frames = len(frame_times)
mean_ms      = sum(frame_times) / total_frames
min_ms       = min(frame_times)
max_ms       = max(frame_times)
std_ms       = (sum((t - mean_ms) ** 2 for t in frame_times) / total_frames) ** 0.5

print(f"\nFrames evaluated : {total_frames}")
print(f"Mean latency     : {mean_ms:.2f} ms  ({1000/mean_ms:.1f} FPS)")
print(f"Min  latency     : {min_ms:.2f} ms")
print(f"Max  latency     : {max_ms:.2f} ms")
print(f"Std  latency     : {std_ms:.2f} ms")

warmup = 10
mean_ms_no_warmup = sum(frame_times[warmup:]) / len(frame_times[warmup:])
print(f"Mean (excl. warmup): {mean_ms_no_warmup:.2f} ms  ({1000/mean_ms_no_warmup:.1f} FPS)")